# 14 - Imbalance Jump Audit

Compare historical imbalance jump-down and jump-up events year by year with the jump frequency implied by the Phase 4 LSMC imbalance process.

Definitions used here match the calibration code:

- imbalance basis: `delta_imb = system_price - day_ahead_price`
- historical jump: `abs(delta_imb) > 2.5 * iterative_sigma_estimate`, using `src.processes.imbalance._filter_jumps`
- jump down / negative jump: historical jump with `delta_imb < 0`
- jump up / positive jump: historical jump with `delta_imb > 0`
- Phase 4 LSMC horizon: 4,320 half-hours, 500 forward paths

In [1]:
from pathlib import Path
import json
import math

import numpy as np
import pandas as pd

import sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.processes.imbalance import _filter_jumps

RAW = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed'

DA_PATH = RAW / 'elexon_da_prices.parquet'
SP_PATH = RAW / 'elexon_sp_prices.parquet'
IMB_PARAMS_PATH = PROCESSED / 'imbalance_params.json'

LSMC_FORWARD_PATHS = 500
LSMC_HORIZON_HH = 4_320
FULL_YEAR_HH = 365 * 48

print(f'Root: {ROOT}')
print(f'DA:   {DA_PATH}')
print(f'SP:   {SP_PATH}')

Root: G:\My Drive\Research\bess_project
DA:   G:\My Drive\Research\bess_project\data\raw\elexon_da_prices.parquet
SP:   G:\My Drive\Research\bess_project\data\raw\elexon_sp_prices.parquet


In [2]:
df_da = pd.read_parquet(DA_PATH)
df_sp = pd.read_parquet(SP_PATH)

merged = df_da[['settlement_date', 'settlement_period', 'price_gbp_mwh']].merge(
    df_sp[['settlement_date', 'settlement_period', 'system_price']],
    on=['settlement_date', 'settlement_period'],
    how='inner',
)
merged['settlement_date'] = pd.to_datetime(merged['settlement_date'])
merged['year'] = merged['settlement_date'].dt.year
merged['delta_imb'] = merged['system_price'] - merged['price_gbp_mwh']
merged = merged[np.isfinite(merged['delta_imb'])].copy()

jump_mask = _filter_jumps(merged['delta_imb'].to_numpy(dtype=float), threshold_sigma=2.5)
merged['is_jump'] = jump_mask
merged['is_negative_jump'] = jump_mask & (merged['delta_imb'] < 0)
merged['is_positive_jump'] = jump_mask & (merged['delta_imb'] > 0)

print(f'Historical observations: {len(merged):,}')
print(f"Date range: {merged['settlement_date'].min().date()} to {merged['settlement_date'].max().date()}")
print(f"Jump HHs: {int(merged['is_jump'].sum()):,}")
print(f"Negative jump HHs: {int(merged['is_negative_jump'].sum()):,}")

Historical observations: 72,258
Date range: 2024-04-01 to 2026-04-25
Jump HHs: 4,363
Negative jump HHs: 32


In [3]:
def _jump_stat(s: pd.Series, mask_col: str, stat: str) -> float:
    x = s[merged.loc[s.index, mask_col]]
    if len(x) == 0:
        return np.nan
    if stat == 'mean':
        return float(x.mean())
    if stat == 'min':
        return float(x.min())
    if stat == 'max':
        return float(x.max())
    if stat == 'median':
        return float(x.median())
    raise ValueError(stat)

hist_year = (
    merged.groupby('year')
    .agg(
        obs_hh=('delta_imb', 'size'),
        jump_hh=('is_jump', 'sum'),
        negative_jump_hh=('is_negative_jump', 'sum'),
        positive_jump_hh=('is_positive_jump', 'sum'),
        mean_negative_jump_gbp_mwh=('delta_imb', lambda s: _jump_stat(s, 'is_negative_jump', 'mean')),
        min_negative_jump_gbp_mwh=('delta_imb', lambda s: _jump_stat(s, 'is_negative_jump', 'min')),
        p50_negative_jump_gbp_mwh=('delta_imb', lambda s: _jump_stat(s, 'is_negative_jump', 'median')),
        mean_positive_jump_gbp_mwh=('delta_imb', lambda s: _jump_stat(s, 'is_positive_jump', 'mean')),
        max_positive_jump_gbp_mwh=('delta_imb', lambda s: _jump_stat(s, 'is_positive_jump', 'max')),
        p50_positive_jump_gbp_mwh=('delta_imb', lambda s: _jump_stat(s, 'is_positive_jump', 'median')),
    )
    .reset_index()
)
hist_year['negative_jump_rate_per_hh'] = hist_year['negative_jump_hh'] / hist_year['obs_hh']
hist_year['negative_jump_rate_per_full_year_hh'] = hist_year['negative_jump_rate_per_hh'] * FULL_YEAR_HH
hist_year['positive_jump_rate_per_hh'] = hist_year['positive_jump_hh'] / hist_year['obs_hh']
hist_year['positive_jump_rate_per_full_year_hh'] = hist_year['positive_jump_rate_per_hh'] * FULL_YEAR_HH

neg = merged.loc[merged['is_negative_jump'], 'delta_imb']
pos = merged.loc[merged['is_positive_jump'], 'delta_imb']
hist_total = pd.DataFrame([{
    'year': 'Historical total',
    'obs_hh': len(merged),
    'jump_hh': int(merged['is_jump'].sum()),
    'negative_jump_hh': int(merged['is_negative_jump'].sum()),
    'positive_jump_hh': int(merged['is_positive_jump'].sum()),
    'mean_negative_jump_gbp_mwh': float(neg.mean()),
    'min_negative_jump_gbp_mwh': float(neg.min()),
    'p50_negative_jump_gbp_mwh': float(neg.median()),
    'mean_positive_jump_gbp_mwh': float(pos.mean()),
    'max_positive_jump_gbp_mwh': float(pos.max()),
    'p50_positive_jump_gbp_mwh': float(pos.median()),
    'negative_jump_rate_per_hh': float(merged['is_negative_jump'].mean()),
    'negative_jump_rate_per_full_year_hh': float(merged['is_negative_jump'].mean() * FULL_YEAR_HH),
    'positive_jump_rate_per_hh': float(merged['is_positive_jump'].mean()),
    'positive_jump_rate_per_full_year_hh': float(merged['is_positive_jump'].mean() * FULL_YEAR_HH),
}])

hist_display = pd.concat([hist_year, hist_total], ignore_index=True)
hist_display


,year,obs_hh,jump_hh,negative_jump_hh,positive_jump_hh,mean_negative_jump_gbp_mwh,min_negative_jump_gbp_mwh,p50_negative_jump_gbp_mwh,mean_positive_jump_gbp_mwh,max_positive_jump_gbp_mwh,p50_positive_jump_gbp_mwh,negative_jump_rate_per_hh,negative_jump_rate_per_full_year_hh,positive_jump_rate_per_hh,positive_jump_rate_per_full_year_hh
0,2024,26370,1069,14,1055,-161.879286,-202.04,-160.02,141.278669,669.212,129.62,0.000531,9.301479,0.040008,700.932878
1,2025,34859,2164,16,2148,-187.286656,-508.19,-128.88,162.012405,2900.000,134.91,0.000459,8.041539,0.061620,1079.576580
2,2026,11029,1130,2,1128,-121.800000,-122.60,-121.80,144.794391,750.000,137.60,0.000181,3.177079,0.102276,1791.872337
3,Historical total,72258,4363,32,4331,-172.078015,-508.19,-137.18,152.477422,2900.000,134.49,0.000443,7.758864,0.059938,1050.113759


In [4]:
with open(IMB_PARAMS_PATH) as fh:
    imb = json.load(fh)

lambda_jump = float(imb['lambda_jump'])
p_pos = float(imb['p_pos'])
p_neg = 1.0 - p_pos
scale_neg = float(imb['jump_scale_neg'])
scale_pos = float(imb['jump_scale_pos'])

# Intended compound-Poisson expectation: E[number of signed jumps] = N * T * lambda * p_sign.
lsmc_expected_neg_jumps_per_path = LSMC_HORIZON_HH * lambda_jump * p_neg
lsmc_expected_pos_jumps_per_path = LSMC_HORIZON_HH * lambda_jump * p_pos
lsmc_expected_neg_jumps_all_paths = LSMC_FORWARD_PATHS * lsmc_expected_neg_jumps_per_path
lsmc_expected_pos_jumps_all_paths = LSMC_FORWARD_PATHS * lsmc_expected_pos_jumps_per_path

# The current chunked simulator stores one jump size when at least one jump arrives in a half-hour.
# This is slightly lower than the compound-Poisson event count when multiple jumps arrive in the same HH.
lsmc_impl_expected_neg_jump_hh_all_paths = (
    LSMC_FORWARD_PATHS * LSMC_HORIZON_HH * (1.0 - math.exp(-lambda_jump)) * p_neg
)
lsmc_impl_expected_pos_jump_hh_all_paths = (
    LSMC_FORWARD_PATHS * LSMC_HORIZON_HH * (1.0 - math.exp(-lambda_jump)) * p_pos
)

lsmc_row = pd.DataFrame([{
    'source': 'Phase 4 LSMC expected, 500 paths x 4,320 HH',
    'obs_hh': LSMC_FORWARD_PATHS * LSMC_HORIZON_HH,
    'negative_jump_hh_or_events': lsmc_expected_neg_jumps_all_paths,
    'positive_jump_hh_or_events': lsmc_expected_pos_jumps_all_paths,
    'negative_jump_hh_impl_approx': lsmc_impl_expected_neg_jump_hh_all_paths,
    'positive_jump_hh_impl_approx': lsmc_impl_expected_pos_jump_hh_all_paths,
    'negative_jump_rate_per_hh': lambda_jump * p_neg,
    'positive_jump_rate_per_hh': lambda_jump * p_pos,
    'negative_jump_rate_per_full_year_hh': lambda_jump * p_neg * FULL_YEAR_HH,
    'positive_jump_rate_per_full_year_hh': lambda_jump * p_pos * FULL_YEAR_HH,
    'expected_negative_per_path_4320hh': lsmc_expected_neg_jumps_per_path,
    'expected_positive_per_path_4320hh': lsmc_expected_pos_jumps_per_path,
    'mean_negative_jump_gbp_mwh': -scale_neg,
    'p50_negative_jump_gbp_mwh': -scale_neg * math.log(2.0),
    'p95_negative_jump_gbp_mwh': -scale_neg * math.log(20.0),
    'p99_negative_jump_gbp_mwh': -scale_neg * math.log(100.0),
    'mean_positive_jump_gbp_mwh': scale_pos,
    'p50_positive_jump_gbp_mwh': scale_pos * math.log(2.0),
    'p95_positive_jump_gbp_mwh': scale_pos * math.log(20.0),
    'p99_positive_jump_gbp_mwh': scale_pos * math.log(100.0),
}])

lsmc_row


,source,obs_hh,negative_jump_hh_or_events,positive_jump_hh_or_events,negative_jump_hh_impl_approx,positive_jump_hh_impl_approx,negative_jump_rate_per_hh,positive_jump_rate_per_hh,negative_jump_rate_per_full_year_hh,positive_jump_rate_per_full_year_hh,expected_negative_per_path_4320hh,expected_positive_per_path_4320hh,mean_negative_jump_gbp_mwh,p50_negative_jump_gbp_mwh,p95_negative_jump_gbp_mwh,p99_negative_jump_gbp_mwh,mean_positive_jump_gbp_mwh,p50_positive_jump_gbp_mwh,p95_positive_jump_gbp_mwh,p99_positive_jump_gbp_mwh
0,"Phase 4 LSMC expected, 500 paths x 4,320 HH",2160000,956.572283,129466.07988,928.265539,125634.939035,0.000443,0.059938,7.758864,1050.113759,1.913145,258.93216,-172.078015,-119.275391,-515.499665,-792.448547,152.477422,105.689295,456.781534,702.184478


In [5]:
comparison = hist_display.rename(columns={
    'year': 'source',
    'negative_jump_hh': 'negative_jump_hh_or_events',
    'positive_jump_hh': 'positive_jump_hh_or_events',
})[[
    'source', 'obs_hh',
    'negative_jump_hh_or_events', 'positive_jump_hh_or_events',
    'negative_jump_rate_per_hh', 'positive_jump_rate_per_hh',
    'negative_jump_rate_per_full_year_hh', 'positive_jump_rate_per_full_year_hh',
    'mean_negative_jump_gbp_mwh', 'min_negative_jump_gbp_mwh', 'p50_negative_jump_gbp_mwh',
    'mean_positive_jump_gbp_mwh', 'max_positive_jump_gbp_mwh', 'p50_positive_jump_gbp_mwh',
]]

comparison = pd.concat([comparison, lsmc_row[comparison.columns.intersection(lsmc_row.columns)]], ignore_index=True)

out_csv = PROCESSED / 'negative_jump_audit.csv'
out_json = PROCESSED / 'negative_jump_audit.json'
comparison.to_csv(out_csv, index=False)
comparison.to_json(out_json, orient='records', indent=2)

print(f'Saved: {out_csv}')
print(f'Saved: {out_json}')

comparison.round({
    'negative_jump_hh_or_events': 1,
    'positive_jump_hh_or_events': 1,
    'negative_jump_rate_per_hh': 6,
    'positive_jump_rate_per_hh': 6,
    'negative_jump_rate_per_full_year_hh': 2,
    'positive_jump_rate_per_full_year_hh': 2,
    'mean_negative_jump_gbp_mwh': 1,
    'min_negative_jump_gbp_mwh': 1,
    'p50_negative_jump_gbp_mwh': 1,
    'mean_positive_jump_gbp_mwh': 1,
    'max_positive_jump_gbp_mwh': 1,
    'p50_positive_jump_gbp_mwh': 1,
})


Saved: G:\My Drive\Research\bess_project\data\processed\negative_jump_audit.csv
Saved: G:\My Drive\Research\bess_project\data\processed\negative_jump_audit.json


,source,obs_hh,negative_jump_hh_or_events,positive_jump_hh_or_events,negative_jump_rate_per_hh,positive_jump_rate_per_hh,negative_jump_rate_per_full_year_hh,positive_jump_rate_per_full_year_hh,mean_negative_jump_gbp_mwh,min_negative_jump_gbp_mwh,p50_negative_jump_gbp_mwh,mean_positive_jump_gbp_mwh,max_positive_jump_gbp_mwh,p50_positive_jump_gbp_mwh
0,2024,26370,14.0,1055.0,0.000531,0.040008,9.30,700.93,-161.9,-202.0,-160.0,141.3,669.2,129.6
1,2025,34859,16.0,2148.0,0.000459,0.061620,8.04,1079.58,-187.3,-508.2,-128.9,162.0,2900.0,134.9
2,2026,11029,2.0,1128.0,0.000181,0.102276,3.18,1791.87,-121.8,-122.6,-121.8,144.8,750.0,137.6
3,Historical total,72258,32.0,4331.0,0.000443,0.059938,7.76,1050.11,-172.1,-508.2,-137.2,152.5,2900.0,134.5
4,"Phase 4 LSMC expected, 500 paths x 4,320 HH",2160000,956.6,129466.1,0.000443,0.059938,7.76,1050.11,-172.1,NaN,-119.3,152.5,NaN,105.7


## Interpretation

The historical calibration sample contains very few negative jump observations compared with positive jump-up observations. This makes the left-tail jump-size estimate model-risky: the mean negative jump scale is materially influenced by a small number of half-hours.

Positive jump-up events dominate the calibrated jump process. That is why `p_pos` in `imbalance_params.json` is close to one, and why LSMC sees many more expected jump-up events than jump-down events.

The Phase 4 LSMC expected counts are not realised counts from a saved path bundle. They are expectations implied by `imbalance_params.json` for the Phase 4 forward setup. To compare realised LSMC jumps path-by-path, save or reload the simulated `PathBundle.delta_imb` used by notebook 12.